# Simple Rerun Lance Replay

A fresh, single-recording Rerun notebook for ContactBench Lance data.

It logs:

- the MANO hand through `rerun.urdf.UrdfTree`,
- animated URDF joint transforms from Lance `hands[0].urdf_dof`,
- only currently contacting balls during playback,
- current-frame contact points,
- current-frame contact normal-force arrows.


In [ ]:
# Optional: install Rerun into the selected notebook kernel.
import importlib.util
import shutil
import subprocess
import sys

if importlib.util.find_spec("rerun") is None:
    uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("uv is not available on PATH")
    subprocess.check_call([uv, "pip", "install", "--python", sys.executable, "rerun-sdk[notebook]"])
    print("Installed rerun-sdk[notebook]. Restart the kernel and run again.")
else:
    print("rerun is available")

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "outputs").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
DATASET_PATHS = sorted((ROOT / "outputs").glob("*.lance"))
DATASET_PATH = next((p for p in DATASET_PATHS if "10s" in p.name), DATASET_PATHS[0] if DATASET_PATHS else None)
HAND_URDF = ROOT / "assets" / "mano_hand_s02" / "urdf" / "mano_hand_s02_full_convex.urdf"
print("repo", ROOT)
print("dataset", DATASET_PATH)
print("hand_urdf", HAND_URDF)
if DATASET_PATH is None:
    raise RuntimeError("No outputs/*.lance dataset found")

In [ ]:
from typing import Any

import lance
import numpy as np
import rerun as rr
import rerun.blueprint as rrb
from rerun.urdf import UrdfTree

BALL_RADIUS = 0.03
FORCE_SCALE = 0.25
URDF_DOF_NAMES = [
    "ARTx", "ARTy", "ARTz", "ARRx", "ARRy", "ARRz",
    "j1_thumb_cmc_abd", "j1_thumb_cmc_flex", "j1_thumb_ip", "j1_thumb_mcp",
    "j2_index_dip", "j2_index_mcp_abd", "j2_index_mcp_flex", "j2_index_pip",
    "j3_middle_dip", "j3_middle_mcp_abd", "j3_middle_mcp_flex", "j3_middle_pip",
    "j4_ring_dip", "j4_ring_mcp_abd", "j4_ring_mcp_flex", "j4_ring_pip",
    "j5_pinky_dip", "j5_pinky_mcp_abd", "j5_pinky_mcp_flex", "j5_pinky_pip",
]
def as_xyz(value: Any) -> np.ndarray:
    arr = np.asarray(value if value is not None else [], dtype=np.float32)
    if arr.size == 0:
        return np.zeros((0, 3), dtype=np.float32)
    return arr.reshape((-1, 3))


def load_row(path: Path) -> dict[str, Any]:
    rows = lance.dataset(path).to_table(limit=1).to_pylist()
    if not rows:
        raise RuntimeError(f"empty Lance dataset: {path}")
    return rows[0]


def object_arrays(row: dict[str, Any]) -> tuple[list[str], list[np.ndarray]]:
    meta = row.get("trajectory_metadata") or {}
    names = [str(x) for x in (meta.get("object_names") or [])]
    arrays = []
    for i, obj in enumerate(row.get("objects") or []):
        arrays.append(as_xyz((obj or {}).get("pos")))
        if i >= len(names):
            names.append(f"object_{i:03d}")
    return names, arrays


def hand_urdf_dof(row: dict[str, Any]) -> np.ndarray:
    hands = row.get("hands") or []
    if not hands:
        return np.zeros((0, len(URDF_DOF_NAMES)), dtype=np.float32)
    return np.asarray((hands[0] or {}).get("urdf_dof") or [], dtype=np.float32).reshape((-1, len(URDF_DOF_NAMES)))


def log_hand_urdf(rec: rr.RecordingStream, row: dict[str, Any], total_frames: int) -> int:
    if not HAND_URDF.exists():
        raise RuntimeError(f"missing hand URDF: {HAND_URDF}")

    # Use entity_path_prefix="world" so imported mesh paths are clean, e.g.
    # /world/mano_hand/visual_geometries/palm/visual_0.
    tree = UrdfTree.from_file_path(
        HAND_URDF,
        entity_path_prefix="world",
        frame_prefix=None,
        static_transform_entity_path="world/tf_static",
    )
    tree.log_urdf_to_recording(recording=rec)

    joints = {joint.name: joint for joint in tree.joints()}
    dof = hand_urdf_dof(row)
    frames = min(total_frames, len(dof))
    for frame_idx in range(frames):
        rr.set_time("frame", sequence=frame_idx, recording=rec)
        values = dof[frame_idx]
        for joint_idx, name in enumerate(URDF_DOF_NAMES):
            joint = joints.get(name)
            if joint is None:
                continue
            # Important: each joint transform must be logged to a distinct entity.
            # Logging all Transform3D values to one path overwrites all but the last
            # transform on a frame.
            rr.log(f"world/tf/{name}", joint.compute_transform(float(values[joint_idx]), clamp=False), recording=rec)
    return frames
def contacts_by_frame(row: dict[str, Any]) -> list[tuple[np.ndarray, np.ndarray, set[str]]]:
    frames = []
    for frame_contacts in row.get("contact") or []:
        pts = []
        forces = []
        touching_objects = set()
        for entry in frame_contacts or []:
            object_name = str((entry or {}).get("object_name") or "")
            if object_name:
                touching_objects.add(object_name)
            for pair in (entry or {}).get("contact_pairs") or []:
                pos = (pair or {}).get("pos_world")
                if pos is None:
                    continue
                pts.append(pos)
                forces.append((pair or {}).get("force_normal") or [0.0, 0.0, 0.0])
        frames.append((as_xyz(pts), as_xyz(forces), touching_objects))
    return frames

def color_for_index(i: int) -> list[int]:
    return [80 + (i * 47) % 175, 160 + (i * 29) % 95, 255 - (i * 31) % 175]


def log_in_floating_base(path: str, *entities: Any, static: bool = False, recording: rr.RecordingStream) -> None:
    rr.log(path, rr.CoordinateFrame("floating_base"), *entities, static=static, recording=recording)


def log_replay(row: dict[str, Any]) -> rr.RecordingStream:
    rec = rr.RecordingStream("contactbench_lance_replay_simple", make_default=False)
    rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, rr.CoordinateFrame("floating_base"), static=True, recording=rec)

    names, objects = object_arrays(row)
    name_to_index = {name: idx for idx, name in enumerate(names)}
    contacts = contacts_by_frame(row)
    total_frames = int((row.get("trajectory_metadata") or {}).get("total_frames", len(row.get("timestamp") or [])))
    total_frames = min(total_frames, max((len(x) for x in objects), default=0), len(contacts))
    hand_frames = log_hand_urdf(rec, row, total_frames)

    # Animated layers: only objects that are in contact on the current frame.
    contact_frames = 0
    contact_points_total = 0
    max_touching_objects = 0
    for frame_idx in range(total_frames):
        rr.set_time("frame", sequence=frame_idx, recording=rec)
        pts, forces, touching_objects = contacts[frame_idx]

        touching_indices = sorted(
            idx for name in touching_objects
            for idx in [name_to_index.get(name, -1)]
            if idx >= 0 and frame_idx < len(objects[idx])
        )
        centers = np.asarray([objects[idx][frame_idx] for idx in touching_indices], dtype=np.float32).reshape((-1, 3))
        colors = [color_for_index(idx) for idx in touching_indices]
        max_touching_objects = max(max_touching_objects, len(centers))
        if len(centers):
            log_in_floating_base("world/animated/touching_balls", rr.Points3D(centers, colors=colors, radii=[BALL_RADIUS] * len(centers)), recording=rec)
        else:
            rr.log("world/animated/touching_balls", rr.Clear(recursive=True), recording=rec)

        if len(pts):
            contact_frames += 1
            contact_points_total += len(pts)
            log_in_floating_base("world/animated/contact_points", rr.Points3D(pts, colors=[[255, 0, 0]], radii=[0.006] * len(pts)), recording=rec)
            log_in_floating_base("world/animated/contact_forces", rr.Arrows3D(origins=pts, vectors=forces * FORCE_SCALE, colors=[[255, 230, 0]], radii=[0.003] * len(pts)), recording=rec)
        else:
            rr.log("world/animated/contact_points", rr.Clear(recursive=True), recording=rec)
            rr.log("world/animated/contact_forces", rr.Clear(recursive=True), recording=rec)

    print({
        "frames_logged": total_frames,
        "objects_total": len(objects),
        "hand_urdf_frames": int(hand_frames),
        "showing": "current-frame touching balls only; no all-ball debug layer",
        "force_scale": FORCE_SCALE,
        "contact_frames": contact_frames,
        "contact_points_total": int(contact_points_total),
        "max_touching_objects_in_frame": int(max_touching_objects),
        "paths": [
            "world/mano_hand",
            "world/tf",
            "world/animated/touching_balls",
            "world/animated/contact_points",
            "world/animated/contact_forces",
        ],
    })
    return rec

row = load_row(DATASET_PATH)
recording = log_replay(row)

In [ ]:
blueprint = rrb.Blueprint(
    rrb.Spatial3DView(origin="world", contents="world/**", name="ContactBench Replay"),
    collapse_panels=False,
)
rr.notebook_show(width=2000, height=1000, blueprint=blueprint, recording=recording)